## A full business solution
### Now we will take our project from Day 1 to the next level
### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment 
# with (llms) in the command prompt

import os 
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
# Initialize and constraints

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API Key looks good so far.")
else:
    print("There might be a problem with your API key. Please visit the troubleshooting notebook!")

MODEL = "gpt-5-nano"
openai_client = OpenAI()

API Key looks good so far.


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include
in a brochure about the company, such as about page, or a company page, or
Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url.careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):

    user_prompt = f"""
Here is the list of links on the website {url}.
Please decide which of these are relevant web links for a brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links

Links (Some might be relative links):

"""
        
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com.
Please decide which of these are relevant web links for a brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links

Links (Some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.co

In [7]:
def select_relevant_links(url):

    response = openai_client.chat.completions.create(
        model= MODEL,
        messages= [
            {'role': 'system', 'content': link_system_prompt},
            {'role': 'user', 'content': get_links_user_prompt(url)}
        ],

        response_format= {'type': "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'associated company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'}]}

In [11]:

select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'product page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

### Second step: make the brochure!
Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    for link in relevant_links['links']:
        result += f"\n\n### Link: Type: {link['type']}\n" 
        # result += f"\n\n### Link: {link['url']}\n Type: {link['type']}\n" # to print link also
        result += fetch_website_contents(link["url"])
    
    return result

In [13]:
fetch_page_and_all_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n5 days ago\n•\n769k\n•\n919\nQwen/Qwen3.5-9B\nUpdated\n3 days ago\n•\n172k\n•\n383\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 6 hours ago\n•\n674k\n•\n491\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n407k\n•\n569\nQwen/Qwen3.5-0.8B\nUpdated\n2 days ago\n•\n93.4k\n•\n225\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n323\nOmni Video Factory\n🏆\n323\ntext to video, image to video, video extend\nRunning\non\nZero\nFeatured\n1.8k\nQwen Image Multiple Angles 3D Camera\n🎥\n1.8k\nChange the camera angle of a photo with AI\nRunning\non\nZero\nMCP\n1.07k\nWan2.2 14B Preview\

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from 
a company website and creates a short brochure about the company for prospective 
customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the 
information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown 
without code blocks.\n\n
"""
    
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown \nwithout code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n5 days ago\n•\n769k\n•\n919\nQwen/Qwen3.5-9B\nUpdated\n3 days ago\n•\n172k\n•\n383\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 6 hours ago\n•\n674k\n•\n491\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n407k\n•\n569\nQwen/Qwen3.5-0.8B\nUpdated\n2 days ago\n•\n93.4k\n•\n225\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n323\nOmni Video Factory\n🏆\n323

In [17]:
def create_brochure(company_name, url):
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the leading collaboration platform for the machine learning (ML) community, empowering engineers, scientists, and AI enthusiasts worldwide to create, share, and experiment with open-source ML models, datasets, and applications. By providing a central hub—the Hugging Face Hub—users gain access to over **2 million models**, **500,000+ datasets**, and **1 million+ AI-powered applications**, covering a diverse range of modalities including text, image, video, audio, and 3D.

As the heart of the AI revolution, Hugging Face fosters an open, ethical, and transparent AI future, supported by a fast-growing community and a talented science team pushing the boundaries of technology.

---

## Platform Features

- **Model & Dataset Sharing:** Host and collaborate on unlimited public models and datasets.
- **Spaces:** Run and share AI applications created by the community, leveraging easy-to-use UI.
- **Open Source Stack:** Use Hugging Face’s open-source libraries and tools to accelerate development in NLP, vision, speech, and beyond.
- **Multimodal AI:** Explore applications spanning multiple modalities such as text-to-speech, text-to-image, video generation, and 3D camera angle manipulation.
- **Build Your Profile:** Share your work and build an ML portfolio to showcase your skills to the community.

---

## Enterprise Solutions

Hugging Face also offers **Team & Enterprise plans** designed to scale organizations efficiently with the world’s leading AI platform. These plans include:

- **Advanced Security**: Single Sign-On (SSO), region-based data management, audit logging, and granular resource group permissions.
- **Scalability & Performance**: Access to advanced compute options such as ZeroGPU with 5x quota boosts.
- **Collaboration Tools**: Private dataset viewers and additional private storage per user.
- **Monitoring & Controls**: Centralized token management, usage analytics dashboard, and spending controls for inference providers.
- Flexible contract options tailored to teams of all sizes to accelerate AI development with enterprise-grade support.

---

## Company Culture & Community

Hugging Face thrives on a community-first philosophy, championing openness, collaboration, and inclusivity. They empower the next generation of ML engineers and scientists by fostering a vibrant global network where knowledge sharing and innovation happen organically.

The company supports ethical AI development and encourages transparent practices to ensure the responsible use of AI technologies.

Community engagement is a core pillar of Hugging Face’s identity with active forums, open-source projects, and resources available to all.

---

## Careers & Opportunities

Joining Hugging Face means being part of a pioneering team that is shaping the future of AI. The company seeks diverse talent who are passionate about machine learning, open source, and community building. Career opportunities span research science, software engineering, product management, data science, and more.

Employees benefit from a collaborative environment that values creativity, continuous learning, and impact. Hugging Face prioritizes building tools that amplify human capabilities through responsible AI.

Explore current job openings and become part of the AI revolution [here](https://huggingface.co/careers).

---

## Why Choose Hugging Face?

- **Industry Leader**: One of the most widely used open-source ML libraries worldwide.
- **Unmatched Collaboration**: A thriving ecosystem of millions of users sharing and advancing AI technology.
- **Comprehensive Platform**: From models and datasets to hosting AI applications and enterprise-grade solutions.
- **Innovation at the Edge**: Continuous development of cutting-edge ML tools and research.

---

## Contact & Explore

- **Website**: [huggingface.co](https://huggingface.co)
- **Community & Forum**: Engage with fellow ML practitioners.
- **Documentation & Resources**: Comprehensive guides to get started quickly.
- **Social Media**: Follow Hugging Face on GitHub, Twitter, LinkedIn, and Discord for the latest updates.

---

**Hugging Face – Building the future of AI, together.**

### Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI, with the familiar typewriter animation

In [22]:
from IPython.display import update_display
def stream_brochure(company_name, url):
    
    stream = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= [
            {'role': 'system', 'content': brochure_system_prompt},
            {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
        ],
        stream= True
    )

    response = ""
    display_handle = display(Markdown(""), display_id= True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id= display_handle.display_id)


In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. As the premier collaboration platform for the global machine learning community, Hugging Face enables engineers, scientists, and developers to create, discover, and share machine learning models, datasets, and AI applications. Their central hub fosters an open and ethical AI future by empowering individuals and teams to collaborate on a massive scale.

## What We Offer

- **Hugging Face Hub:** A central place to host and collaborate on unlimited public machine learning models, datasets, and applications.
- **Models:** Access to over 2 million diverse state-of-the-art models, including trending models like Qwen series.
- **Datasets:** Explore and contribute to more than 500,000 datasets across modalities.
- **Spaces:** Share and run AI-powered applications covering text, image, video, audio, and even 3D.
- **Open-source Stack:** Accelerate machine learning development with cutting-edge open-source tools and libraries.

## Key Features

- **Community-Driven:** Join a fast-growing, vibrant community collaborating globally on open-source AI projects.
- **Multimodal AI:** Develop and experiment with AI models beyond text—explore video, audio, image, and 3D ML applications.
- **Portfolio Building:** Showcase your projects and build your professional machine learning profile.
- **Enterprise Solutions:** Access paid compute resources and enterprise-grade solutions that scale to meet business needs.

## Customers & Users

Hugging Face serves a broad spectrum of users including machine learning engineers, data scientists, researchers, AI enthusiasts, startups, and enterprises. The platform nurtures talent and innovation across academia, industry, and independent developers.

## Company Culture

Hugging Face is founded on values of openness, collaboration, and ethical AI. The company champions an inclusive and supportive environment where ideas flourish through shared knowledge. Its culture celebrates innovation, respect for data and user privacy, and building AI technologies that benefit everyone.

## Careers & Opportunities

Join Hugging Face if you are passionate about AI and want to contribute to the future of machine learning. The company offers exciting opportunities for engineers, researchers, data scientists, and community advocates.

- Be part of a pioneering team shaping the open-source AI landscape.
- Collaborate with a distributed, global team of diverse and talented professionals.
- Work on impactful products that reach millions of users worldwide.
- Embrace continuous learning and growth in a fast-evolving field.

Visit the Hugging Face website to explore current job openings and join the AI revolution.

---

**Hugging Face**  
The AI Community Building the Future  
Website: https://huggingface.co  
Connect, Collaborate, Create – Build your future with AI today!